# ingest_file_autoloader\nGenerated from 01_ingestion/ingest_file_autoloader.py for Databricks notebook import.\n

In [ ]:
"""File ingestion adapter for Databricks Auto Loader."""\n\nfrom __future__ import annotations\n\n\ndef _build_tracking_paths(global_config: dict, source: dict) -> tuple[str, str]:\n    dbx_cfg = global_config.get("databricks", {})\n    checkpoint_root = dbx_cfg.get("checkpoint_root")\n    schema_root = dbx_cfg.get("schema_tracking_root")\n    tenant = source.get("tenant")\n    product = source.get("product_name")\n    source_system = source.get("source_system")\n    source_entity = source.get("source_entity")\n\n    required = {\n        "checkpoint_root": checkpoint_root,\n        "schema_tracking_root": schema_root,\n        "tenant": tenant,\n        "product_name": product,\n        "source_system": source_system,\n        "source_entity": source_entity,\n    }\n    missing = [k for k, v in required.items() if not v]\n    if missing:\n        raise ValueError(f"Missing tracking path configuration: {missing}")\n\n    path_suffix = f"{tenant}/{product}/{source_system}/{source_entity}"\n    checkpoint_location = f"{checkpoint_root.rstrip('/')}/{path_suffix}"\n    schema_location = f"{schema_root.rstrip('/')}/{path_suffix}"\n    return schema_location, checkpoint_location\n\n\ndef build_autoloader_options(source: dict, global_config: dict, source_options: dict | None = None) -> dict:\n    """Build the full Auto Loader option dict for a given source.\n\n    Extra keys from ``source_options`` are merged last so per-source overrides\n    win over framework defaults.  The caller is responsible for separating\n    streaming options (``cloudFiles.*``) from write options (``checkpointLocation``).\n    """\n    source_path = source.get("source_path")\n    if not source_path:\n        raise ValueError("source_path is required for FILE source types")\n\n    schema_location, checkpoint_location = _build_tracking_paths(global_config, source)\n    file_defaults = global_config.get("connections", {}).get("file_defaults")\n    if not file_defaults:\n        raise ValueError("connections.file_defaults must be configured in global config")\n\n    required_defaults = {"format", "infer_column_types", "schema_evolution_mode"}\n    missing_defaults = required_defaults - set(file_defaults)\n    if missing_defaults:\n        raise ValueError(f"file_defaults must define: {sorted(missing_defaults)}")\n\n    format_value = source.get("source_format") or file_defaults["format"]\n    infer_types = str(file_defaults["infer_column_types"]).lower()\n    schema_evolution_mode = file_defaults["schema_evolution_mode"]\n\n    options: dict = {\n        "cloudFiles.format": format_value,\n        "cloudFiles.schemaLocation": schema_location,\n        "cloudFiles.inferColumnTypes": infer_types,\n        "cloudFiles.schemaEvolutionMode": schema_evolution_mode,\n        "path": source_path,\n        "checkpointLocation": checkpoint_location,\n    }\n\n    # Per-source overrides from source_options_json (last-write wins).\n    for k, v in (source_options or {}).items():\n        if isinstance(v, (str, int, float, bool)):\n            options[k] = str(v)\n\n    return options\n\n\ndef ingest_file_stream(spark, source: dict, global_config: dict, source_options: dict | None = None):\n    """Start an Auto Loader structured streaming read and return the streaming DataFrame.\n\n    The returned DataFrame is *not* yet writing; callers must call\n    ``write_file_stream_to_landing`` or attach their own write stream.\n    """\n    options = build_autoloader_options(source=source, global_config=global_config, source_options=source_options)\n\n    # Separate cloudFiles read options from write-side options.\n    read_options = {k: v for k, v in options.items() if k not in {"path", "checkpointLocation"}}\n\n    reader = spark.readStream.format("cloudFiles").options(**read_options)\n\n    # Optional explicit schema — set schema_ddl in source_options_json to skip inference.\n    schema_ddl = (source_options or {}).get("schema_ddl", "")\n    if schema_ddl:\n        from pyspark.sql.types import StructType\n        reader = reader.schema(StructType.fromDDL(schema_ddl))\n\n    return reader.load(options["path"])\n\n\ndef write_file_stream_to_landing(\n    spark,\n    streaming_df,\n    source: dict,\n    global_config: dict,\n    source_options: dict | None = None,\n) -> None:\n    """Write a streaming DataFrame to the landing Delta table.\n\n    Uses ``availableNow=True`` so the query processes all available files then\n    stops — equivalent to a micro-batch full-catch-up, friendly for scheduled jobs.\n\n    Raises ``RuntimeError`` if the streaming query terminates with an exception,\n    wrapping the cause with the source identifier for easier debugging.\n    """\n    options = build_autoloader_options(source=source, global_config=global_config, source_options=source_options)\n    checkpoint_location = options["checkpointLocation"]\n    landing_table = source.get("landing_table", "")\n    if not landing_table:\n        raise ValueError("landing_table is required in source metadata for Auto Loader write")\n\n    merge_schema = str((source_options or {}).get("mergeSchema", "true")).lower()\n\n    try:\n        query = (\n            streaming_df.writeStream.format("delta")\n            .outputMode("append")\n            .option("checkpointLocation", checkpoint_location)\n            .option("mergeSchema", merge_schema)\n            .trigger(availableNow=True)\n            .toTable(landing_table)\n        )\n        query.awaitTermination()\n        if query.exception():\n            raise RuntimeError(query.exception())\n    except Exception as exc:\n        source_id = f"{source.get('source_system', '?')}.{source.get('source_entity', '?')}"\n        raise RuntimeError(\n            f"Auto Loader streaming write failed for {source_id}: {exc}"\n        ) from exc\n